# Notebook 02e — Test-Retest Reliability of the LLM Classifier

**Purpose.** Report intra-classifier stability (test-retest kappa) for the LLM classification pipeline used in Notebook 02c. F1 against the human reference set (BE = 0.58, ME = 0.71, PSI = 0.29, PI = 0.80) already establishes *convergent validity* — how well the LLM agrees with human coders on average. This notebook adds *intra-classifier stability* — how consistent the LLM is with itself when the same comment is scored twice. Both are standard requirements in modern content-analysis reporting (Krippendorff, 2018) and in current LLM-as-annotator methodology (Ziems et al., 2024).

**Design.**

1. Sample 100 comments from the already-scored corpus, stratified so that each construct has roughly 20 positive-labelled comments and the remaining slots are filled with all-negative comments. Over-sampling positives avoids a trivial-agreement floor caused by the low base rates of BE (16%), ME (13%), PSI (11%), and PI (5%).
2. Re-score all 100 comments using the **identical** system prompt, model version (`claude-sonnet-4-6`), and API parameters used in the original scoring (Notebook 02c).
3. Compute Cohen's kappa on the binary judgments for each construct, plus percent agreement and the 2 × 2 confusion matrix per construct.
4. Emit a summary table and a paste-ready paragraph for the Methods chapter.

**Caveat on model drift.** Anthropic does not publish a fine-grained version history for Claude Sonnet 4.6. A minor drop in kappa could reflect a silent model update between the original scoring and the re-scoring rather than genuine per-run stochasticity. The reported kappa should be read as a conservative lower bound on stability under a fixed model.

## 0. Setup

In [ ]:
import os
import json
import time
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from anthropic import Anthropic
from sklearn.metrics import (
    cohen_kappa_score, confusion_matrix, accuracy_score,
)

# Reads ANTHROPIC_API_KEY from environment (same convention as 02c)
client = Anthropic()

# Quick smoke test on the API connection
try:
    test = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=20,
        messages=[{'role': 'user', 'content': 'Reply with the single word: OK'}],
    )
    print(f'API connection OK. Test response: {test.content[0].text.strip()}')
except Exception as e:
    raise RuntimeError(
        f'API connection failed: {e}\n\n'
        'Check that ANTHROPIC_API_KEY is set in your environment.'
    )

np.random.seed(42)

## 1. Load the scored corpus and reproduce the classification prompt

The system prompt, `build_user_message`, and `classify_one` below are reproduced **verbatim** from Notebook 02c §3 and §4. Any edit here would invalidate the test-retest comparison. If you have updated the prompt in 02c, sync it here first before re-running.

In [ ]:
df = pd.read_csv('comments_scored_llm.csv')
print(f'Original scored corpus: {len(df):,} comments, '
      f'{df["matched_influencer"].nunique()} influencers')

# Confirm the binary columns are present — needed for stratified sampling
required = ['benign_envy_binary', 'malicious_envy_binary',
            'psi_binary', 'purchase_intent_binary']
missing = [c for c in required if c not in df.columns]
assert not missing, f'Missing binary columns in scored corpus: {missing}'

print('Positive base rates in the corpus:')
for c in required:
    print(f'  {c:<28s} {df[c].mean()*100:5.1f}%  '
          f'(n_pos = {int(df[c].sum())})')

In [ ]:
# ──────────────────────────────────────────────────────────────────
# SYSTEM_PROMPT — reproduced verbatim from Notebook 02c §3.
# ──────────────────────────────────────────────────────────────────
SYSTEM_PROMPT = '''You are a careful research coder for a Master's dissertation on social media influencer marketing. Your task is to read a Reddit comment and judge whether it expresses each of four psychological constructs toward a specific focal influencer.

Apply each construct STRICTLY to the focal influencer named below — not to other people mentioned in the comment.

THE FOUR CONSTRUCTS

1. BENIGN ENVY — The commenter expresses upward admiration toward the focal influencer as a person, with an aspirational or identification-oriented tone. They want to BE LIKE the influencer, treat them as a role model, find them inspiring, or admire their style/skill/qualities in an aspirational way.
   POSITIVE examples: "she's goals 😍", "I want to dress like her", "her glow up is so inspiring", "she's everything I want to be", "Hannah does insanely thorough reviews, I like her a lot", "Lauren Mae is great at describing eyeshadows".
   NEGATIVE examples: "this product is great" (about product, not person), "she's nice" (no aspiration), generic mention with no admiration, hostile/sarcastic comments.

2. MALICIOUS ENVY — The commenter expresses hostility, contempt, sarcasm, mockery, accusations of fakeness, dismissiveness, or bitter resentment toward the focal influencer as a person.
   POSITIVE examples: "so fake", "must be nice 🙄", "she's such a fraud", "Jaclyn Hill 2.0 with all her bullshit", "she's exhausting", explicit critiques of personality.
   NEGATIVE examples: criticism of a product (not the person), factual disagreement, neutral observation, comments hostile to someone other than the focal influencer.

3. PSI (PARASOCIAL INTERACTION) — The commenter writes about the focal influencer as if they personally know them — uses nicknames, defends them from criticism, expresses warm familiarity, treats them like a friend.
   POSITIVE examples: "we love her", "leave my girl alone", "Alix would never", "I feel like I grew up with her", "feels like chatting with a friend", defending the influencer in any way.
   NEGATIVE examples: general approval without closeness ("her work is good"), product discussion, hostile remarks.

4. PURCHASE INTENT — The commenter expresses urgent desire to buy a specific product (often in reaction to the influencer's recommendation).
   POSITIVE examples: "where is the link", "take my money", "adding to cart immediately", "I need this RIGHT NOW", "Julia Adams sold me on this", "I get the urge to try them".
   NEGATIVE examples: discussing past purchases without present urge, browsing without intent, anti-consumption remarks.

OUTPUT FORMAT

Return a single JSON object with this exact structure and no other text:

{
  "benign_envy":     {"score": <float 0-1>, "binary": <0 or 1>, "reason": "<one sentence>"},
  "malicious_envy":  {"score": <float 0-1>, "binary": <0 or 1>, "reason": "<one sentence>"},
  "psi":             {"score": <float 0-1>, "binary": <0 or 1>, "reason": "<one sentence>"},
  "purchase_intent": {"score": <float 0-1>, "binary": <0 or 1>, "reason": "<one sentence>"}
}

The "score" reflects how strongly the comment expresses the construct (0.0 = clearly not expressed, 1.0 = unambiguous expression). The "binary" is your 0/1 judgment at threshold 0.5. The "reason" is a brief justification.'''

def build_user_message(focal_influencer: str, body: str) -> str:
    return f'FOCAL INFLUENCER: {focal_influencer}\n\nCOMMENT:\n{body}'


def classify_one(focal_influencer: str, body: str,
                 model: str = 'claude-sonnet-4-6', retries: int = 3):
    """Send one comment to the LLM and parse the JSON response.
    Reproduced verbatim from Notebook 02c §4."""
    user_msg = build_user_message(focal_influencer, body)
    last_err = None
    for attempt in range(retries):
        try:
            resp = client.messages.create(
                model=model,
                max_tokens=600,
                system=SYSTEM_PROMPT,
                messages=[{'role': 'user', 'content': user_msg}],
            )
            text = resp.content[0].text.strip()
            if text.startswith('```'):
                text = text.split('```')[1]
                if text.startswith('json'):
                    text = text[4:].strip()
            return json.loads(text)
        except (json.JSONDecodeError, IndexError) as e:
            last_err = f'Parse error: {e}'
            time.sleep(1 + attempt)
        except Exception as e:
            last_err = f'API error: {e}'
            time.sleep(2 ** attempt)
    raise RuntimeError(f'Failed after {retries} attempts: {last_err}')

## 2. Build the stratified 100-comment test-retest sample

**Design.** Take up to 20 comments per construct where the binary label is 1 (positives), then fill any remaining slots up to 100 with all-negative comments (where every binary label is 0). Overlap between constructs is allowed but each comment appears only once in the sample — an author who was positive on both BE and ME contributes one row, not two.

**Rationale.** The corpus positivity rates are low (5–16%). A pure random sample of 100 would be dominated by all-negative comments, inflating kappa via trivial agreement. Over-sampling positives gives the reliability metric enough range to detect real instability.

In [ ]:
N_TOTAL       = 100
N_PER_POSITIVE = 20     # target per construct

constructs = ['benign_envy_binary', 'malicious_envy_binary',
              'psi_binary', 'purchase_intent_binary']

sample_ids = set()

# Add up to 20 positives per construct, avoiding duplicates
for c in constructs:
    pool = df[(df[c] == 1) & (~df['id'].isin(sample_ids))]
    take = min(N_PER_POSITIVE, len(pool))
    picked = pool.sample(take, random_state=42)['id'].tolist()
    sample_ids.update(picked)
    print(f'  {c:<28s} pool={len(pool):>4}   picked={take}')

# Fill remaining slots with all-negative comments
remaining = N_TOTAL - len(sample_ids)
if remaining > 0:
    all_neg = df[
        (~df['id'].isin(sample_ids))
        & (df['benign_envy_binary'] == 0)
        & (df['malicious_envy_binary'] == 0)
        & (df['psi_binary'] == 0)
        & (df['purchase_intent_binary'] == 0)
    ]
    take_neg = min(remaining, len(all_neg))
    neg_ids = all_neg.sample(take_neg, random_state=42)['id'].tolist()
    sample_ids.update(neg_ids)
    print(f'  all-negative fill                pool={len(all_neg):>4}   '
          f'picked={take_neg}')

sample_df = df[df['id'].isin(sample_ids)].reset_index(drop=True)
print(f'\nFinal sample size: {len(sample_df)}')
print('Positive counts in the sample (should be roughly 20 per construct):')
for c in constructs:
    print(f'  {c:<28s} n_pos = {int(sample_df[c].sum())}')

# Persist the sample IDs so the analysis is reproducible
sample_df[['id']].to_csv('test_retest_sample_ids.csv', index=False)
print('\nSaved: test_retest_sample_ids.csv')

## 3. Re-score the sample through the API

Uses the same `classify_one` function as Notebook 02c. Results are checkpointed every 25 comments to `test_retest_partial.csv` so a mid-run crash does not lose progress. If the checkpoint exists, the cell resumes from where it left off — re-running is idempotent.

In [ ]:
CHECKPOINT = 'test_retest_partial.csv'
SAVE_EVERY = 25
MODEL      = 'claude-sonnet-4-6'

done_ids = set()
if os.path.exists(CHECKPOINT):
    prior = pd.read_csv(CHECKPOINT)
    done_ids = set(prior['id'])
    print(f'Resuming — {len(done_ids)} comments already re-scored.')

todo = sample_df[~sample_df['id'].isin(done_ids)].reset_index(drop=True)
print(f'Comments to re-score this run: {len(todo)}')

buffer_rows = []
all_new = []
for _, row in tqdm(todo.iterrows(), total=len(todo),
                    desc='Re-scoring'):
    try:
        out = classify_one(row['matched_influencer'], row['body'],
                            model=MODEL)
        rec = {
            'id': row['id'],
            'benign_envy_retest':          out['benign_envy']['score'],
            'malicious_envy_retest':       out['malicious_envy']['score'],
            'psi_retest':                  out['psi']['score'],
            'purchase_intent_retest':      out['purchase_intent']['score'],
            'benign_envy_binary_retest':   out['benign_envy']['binary'],
            'malicious_envy_binary_retest': out['malicious_envy']['binary'],
            'psi_binary_retest':           out['psi']['binary'],
            'purchase_intent_binary_retest': out['purchase_intent']['binary'],
        }
        buffer_rows.append(rec)
        all_new.append(rec)

        if len(buffer_rows) >= SAVE_EVERY:
            new_df = pd.DataFrame(buffer_rows)
            if os.path.exists(CHECKPOINT):
                old = pd.read_csv(CHECKPOINT)
                full = (pd.concat([old, new_df], ignore_index=True)
                          .drop_duplicates('id'))
            else:
                full = new_df
            full.to_csv(CHECKPOINT, index=False)
            buffer_rows = []
    except Exception as e:
        print(f'  Skipped id={row["id"]}: {e}')

# Final flush
if buffer_rows:
    new_df = pd.DataFrame(buffer_rows)
    if os.path.exists(CHECKPOINT):
        old = pd.read_csv(CHECKPOINT)
        full = (pd.concat([old, new_df], ignore_index=True)
                  .drop_duplicates('id'))
    else:
        full = new_df
    full.to_csv(CHECKPOINT, index=False)

retest_df = pd.read_csv(CHECKPOINT)
print(f'\nTotal re-scored: {len(retest_df)}/{len(sample_df)}')

## 4. Compute Cohen's kappa per construct

Merge the original scores (from `comments_scored_llm.csv`) with the re-run scores on comment `id`, then compute Cohen's kappa, percent agreement, and a 2×2 confusion matrix for each construct's binary judgment.

In [ ]:
# Merge original with retest
orig_cols = ['id', 'benign_envy_binary', 'malicious_envy_binary',
             'psi_binary', 'purchase_intent_binary',
             'benign_envy', 'malicious_envy', 'psi', 'purchase_intent']
merged = sample_df[orig_cols].merge(retest_df, on='id', how='inner')
print(f'Merged pairs available: {len(merged)}')

pairs = [
    ('benign_envy',     'benign_envy_binary',     'benign_envy_binary_retest'),
    ('malicious_envy',  'malicious_envy_binary',  'malicious_envy_binary_retest'),
    ('psi',             'psi_binary',             'psi_binary_retest'),
    ('purchase_intent', 'purchase_intent_binary', 'purchase_intent_binary_retest'),
]

rows = []
for name, col1, col2 in pairs:
    y1 = merged[col1].astype(int)
    y2 = merged[col2].astype(int)
    kappa = cohen_kappa_score(y1, y2)
    agree = accuracy_score(y1, y2) * 100
    cm    = confusion_matrix(y1, y2, labels=[0, 1])
    n_pos_orig  = int(y1.sum())
    n_pos_retest = int(y2.sum())
    rows.append({
        'construct':       name,
        'n':               len(y1),
        'n_pos_original':  n_pos_orig,
        'n_pos_retest':    n_pos_retest,
        'percent_agree':   round(agree, 1),
        'cohen_kappa':     round(kappa, 3),
        'cm_00':           int(cm[0, 0]),
        'cm_01':           int(cm[0, 1]),
        'cm_10':           int(cm[1, 0]),
        'cm_11':           int(cm[1, 1]),
    })

results = pd.DataFrame(rows)
print('=' * 72)
print('Test-retest reliability — Cohen\'s kappa per construct')
print('=' * 72)
print(results[['construct', 'n', 'n_pos_original', 'n_pos_retest',
               'percent_agree', 'cohen_kappa']].to_string(index=False))
print()
print('Confusion matrix rows are (original x retest) with '
      '00/01/10/11 = TN/FP/FN/TP framing:')
print(results[['construct', 'cm_00', 'cm_01', 'cm_10', 'cm_11']]
      .to_string(index=False))

In [ ]:
# ──────────────────────────────────────────────────────────────────
# Continuous-score agreement — Pearson correlation on the raw 0-1
# scores. This complements the binary-kappa view and is useful if
# any construct has borderline scores near the 0.5 threshold.
# ──────────────────────────────────────────────────────────────────
cont_rows = []
for name in ['benign_envy', 'malicious_envy', 'psi', 'purchase_intent']:
    y1 = merged[name].astype(float)
    y2 = merged[f'{name}_retest'].astype(float)
    r = np.corrcoef(y1, y2)[0, 1]
    mad = np.mean(np.abs(y1 - y2))
    cont_rows.append({'construct': name,
                       'pearson_r': round(r, 3),
                       'mean_abs_diff': round(mad, 3)})

cont_df = pd.DataFrame(cont_rows)
print('Continuous-score agreement (0–1 scale):')
print(cont_df.to_string(index=False))

In [ ]:
# Save the merged pairs and the summary table
merged.to_csv('test_retest_paired_scores.csv', index=False)
results.to_csv('test_retest_kappa_summary.csv', index=False)
cont_df.to_csv('test_retest_continuous_summary.csv', index=False)
print('Saved: test_retest_paired_scores.csv, '
      'test_retest_kappa_summary.csv, '
      'test_retest_continuous_summary.csv')

## 5. Paste-ready paragraph for the dissertation Methods chapter

The cell below composes a compact, dissertation-ready paragraph summarising the test-retest procedure and results. Copy it into the Methods chapter after the F1-validation subsection.

In [ ]:
# Build the summary paragraph — values interpolated at runtime
kappa_line = ', '.join([
    f"{name.replace('_', ' ')} = {k:.2f}"
    for name, k in zip(results['construct'], results['cohen_kappa'])
])
agree_line = ', '.join([
    f"{name.replace('_', ' ')} = {a:.0f}%"
    for name, a in zip(results['construct'], results['percent_agree'])
])

paragraph = (
    f'Intra-classifier reliability. To assess the stability of the LLM '
    f'classifier across repeated runs, a stratified random sample of '
    f'{len(merged)} comments was re-scored using the identical system '
    f'prompt, model version (claude-sonnet-4-6), and API parameters used '
    f'in the original classification (Notebook 02c). The sample was '
    f'stratified to include approximately 20 positively-labelled '
    f'comments per construct plus an all-negative reference group, in '
    f'order to prevent trivial-agreement inflation caused by the low '
    f'positive-class base rates (5–16%). Cohen\'s kappa between the '
    f'original and re-run binary judgments was {kappa_line}. Percent '
    f'agreement was {agree_line}. Continuous-score agreement '
    f'(Pearson r and mean absolute difference) is reported alongside. '
    f'These figures complement the F1-based convergent validity '
    f'reported above and support the claim that the LLM classifier '
    f'produces stable scores under repeated exposure to the same '
    f'input. A caveat is noted regarding potential silent model updates '
    f'on the Anthropic side; the reported kappa values should therefore '
    f'be read as a conservative lower bound on intra-classifier '
    f'stability under a fixed model version.'
)

print('=' * 72)
print('METHODS CHAPTER — PASTE-READY PARAGRAPH')
print('=' * 72)
print(paragraph)

## 6. Interpretation guide

Common conventions for Cohen's kappa (Landis & Koch, 1977):

| Kappa range | Interpretation |
|---|---|
| < 0.00 | Poor (worse than chance) |
| 0.00 – 0.20 | Slight |
| 0.21 – 0.40 | Fair |
| 0.41 – 0.60 | Moderate |
| 0.61 – 0.80 | Substantial |
| 0.81 – 1.00 | Almost perfect |

**What to expect.** For a well-behaved LLM classifier at `temperature=0`, most constructs should return kappa ≥ 0.80. If a construct comes back < 0.60, that is a substantive stability problem worth flagging in the Limitations chapter — the LLM is not just disagreeing with humans on that construct, it is disagreeing with *itself* between runs. Expect PSI to be the most likely candidate for a lower kappa, consistent with its F1 = 0.29 already reported. If BE, ME, or PI show low kappa, that is a new finding not previously documented and should be added to the Discussion.

**If kappa is disappointing.** Two remedies to consider:

1. Verify the prompt in this notebook exactly matches the prompt in Notebook 02c. Any drift in the prompt would cause disagreement that is prompt-drift, not stochasticity.
2. Consider whether Anthropic has issued a silent model update between the original scoring and now. If your original scoring was more than a few weeks ago, factor this into the write-up as noted in Section 0.